In [ ]:
import os
import sys
from pathlib import Path

# En Serverless, indicar la raíz absoluta del proyecto completo en /Workspace/...
# Puede fijarse aquí o mediante RISKVISION_PROJECT_ROOT; no debe apuntar a notebooks/.
PROJECT_ROOT_OVERRIDE = os.environ.get("RISKVISION_PROJECT_ROOT")
if PROJECT_ROOT_OVERRIDE:
    project_root = Path(PROJECT_ROOT_OVERRIDE).expanduser()
    if not project_root.is_absolute():
        raise ValueError("RISKVISION_PROJECT_ROOT debe ser una ruta absoluta")
    project_root = project_root.resolve()
else:
    # Compatibilidad local: iniciar desde la raíz o desde notebooks/.
    candidates = (Path.cwd().resolve(), Path.cwd().resolve().parent)
    project_root = next((p for p in candidates if (p / "src" / "proyecto").is_dir()), None)
if project_root is None or not (project_root / "src" / "proyecto" / "config.py").is_file():
    raise RuntimeError("No se encuentra src/proyecto: configura RISKVISION_PROJECT_ROOT con la raíz absoluta")
source_dir = str(project_root / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
import proyecto
if Path(proyecto.__file__).resolve().parent != project_root / "src" / "proyecto":
    raise RuntimeError("El kernel tiene cargada otra copia de proyecto; reinícialo y revisa la raíz")
print("Raíz verificada:", project_root)
print("Módulos:", Path(proyecto.__file__).resolve().parent)
print("Python:", sys.version.split()[0], "| plataforma:", sys.platform)


# 06 · Optimización con seguimiento (tracking) — RiskVision

Guía: material-del-curso/kit/guia-modelado-y-optimizacion.md, secciones 3 y 4. Grupo 6.

Buscar mejores hiperparámetros es probar configuraciones y registrar qué ocurrió en cada ensayo. Se reutiliza src/proyecto/models/tuning.py, sin duplicar la búsqueda en el notebook.

1. Se optimiza **LightGBM con el paquete E**. La regresión logística con E permanece como referencia sin afinar.
2. Se fijan antes el espacio, las semillas, los pliegues y un presupuesto inicial de **12 ensayos**; los fallidos también consumen presupuesto.
3. Se maximiza **ROC AUC**. El mejor valor de búsqueda tiene optimismo de selección y no demuestra mejora.
4. Se carga únicamente el entrenamiento original. El 25 % queda fuera de Optuna como reserva interna. Esa reserva ya participó indirectamente en la elección de variables y modelos de las etapas 04 y 05: **no es una evaluación independiente**. La prueba externa original también fue consultada anteriormente y no se utiliza aquí.

**Estado de esta entrega:** integración preparada, sin ejecutar la búsqueda real. RUN_OPTIMIZATION permanece en False hasta autorización. Las celdas de resultados mostrarán que están pendientes; no se guardan modelos ni parámetros definitivos.

### Preparación de Databricks Serverless (antes de activar la búsqueda)

- Sincronizar el proyecto completo y seleccionar la rama revisada. Configurar PROJECT_ROOT_OVERRIDE con la ruta absoluta /Workspace/... del proyecto. La detección por cwd se conserva solo como comodidad local.
- Configurar TRAIN_PATH_OVERRIDE si el entrenamiento está en otro lugar, por ejemplo /Volumes/<catalogo>/<esquema>/<volumen>/train.joblib. Son rutas de Python, no URI dbfs:. Se comprueban nombre, existencia, dimensiones, objetivo e ID; la huella identifica la copia utilizada.
- Elegir el entorno de Serverless desde su panel Environment. Propuesta mínima de partida, con versiones verificadas localmente en Python 3.12: pandas==3.0.6, numpy==2.5.4, scipy==1.18.1, scikit-learn==1.9.1, joblib==1.6.0, lightgbm==4.7.0, xgboost==3.4.1, optuna==5.0.0, mlflow==3.17.1, databricks-sdk==0.153.0 y python-dotenv==1.2.4. XGBoost es necesario porque candidates.py lo importa, aunque aquí no se entrena. Las dependencias transitivas las resuelve el instalador. No hace falta instalar Streamlit, SHAP, xlrd, herramientas de notebooks o pytest para esta ejecución.
- Esta propuesta no es una compatibilidad certificada con Serverless: verificar Python, arquitectura, resolución de dependencias e imports en el entorno real. Aprovechar las bibliotecas proporcionadas por Databricks cuando sean compatibles; no reemplazar indiscriminadamente el entorno base ni instalar todo requirements.txt. Registrar las versiones efectivas.
- Configurar TRACKING_MODE=databricks y, si corresponde, EXPERIMENT_DIR mediante la configuración Python de la sesión. No se necesitan tokens pegados en el notebook. Los imports y la consulta del servidor deben funcionar antes de crear un experimento o iniciar Optuna. No activar autologging adicional que exporte modelos durante esta etapa.
- La recuperación de estudios y el registro progresivo siguen fuera de alcance: los ensayos terminados se registran progresivamente. Los informes son provisionales y pertenecen a una ejecución identificada; no contienen resultados de otras ramas. La comparación y sus limitaciones estadísticas se conservan.


## 0. Cargar los datos de entrenamiento

Se lee exclusivamente data/processed/train.joblib. Se verifican 24.000 clientes, 25 columnas, ID válidos y objetivo binario. make_xy elimina ID y objetivo de los predictores. La huella del archivo permite identificar el dato usado; no se modifica.


In [ ]:
import hashlib
import joblib
import numpy as np
import pandas as pd

from proyecto.config import ID_COL, TARGET, PROCESSED_DIR
from proyecto.data.split import assert_unique_ids, assert_disjoint_ids, make_xy, split_stratified

# Ruta absoluta opcional, también configurable al editar TRAIN_PATH_OVERRIDE.
TRAIN_PATH_OVERRIDE = os.environ.get("RISKVISION_TRAIN_PATH")
TRAIN_PATH = Path(TRAIN_PATH_OVERRIDE).expanduser() if TRAIN_PATH_OVERRIDE else PROCESSED_DIR / "train.joblib"
if not TRAIN_PATH.is_absolute():
    raise ValueError("RISKVISION_TRAIN_PATH debe ser una ruta absoluta")
TRAIN_PATH = TRAIN_PATH.resolve()
if TRAIN_PATH.name != "train.joblib":
    raise ValueError("Esta etapa admite únicamente el archivo de entrenamiento train.joblib")
print("Archivo de entrenamiento:", TRAIN_PATH)
if not TRAIN_PATH.is_file():
    raise FileNotFoundError("Falta el entrenamiento local; completa primero la preparación de datos")
train = joblib.load(TRAIN_PATH)
assert train.shape == (24_000, 25), f"Dimensiones inesperadas: {train.shape}"
assert_unique_ids(train)
X_all, y_all = make_xy(train)
assert X_all.shape == (24_000, 23)
TRAIN_SHA256 = hashlib.sha256(TRAIN_PATH.read_bytes()).hexdigest()
print(f"Entrenamiento original: {len(train):,} clientes; {X_all.shape[1]} predictores")
print(f"Tasa de incumplimiento: {y_all.mean():.2%}")
print("SHA256 del entrenamiento:", TRAIN_SHA256)


## 1. Pipeline y métrica (los del torneo)

Se mantienen las 23 variables originales y las cinco derivadas del paquete E: utilization_mean, pending_balance_last, delay_streak, n_zero_payments y log_limit. Cada pliegue recibe un pipeline nuevo; medianas, categorías y escalas se aprenden solo de su entrenamiento.

La comparación principal es LightGBM de partida frente a LightGBM optimizado. La logística con E se construye como referencia sin ajustarla ni optimizarla en esta fase. ROC AUC se maximiza; average precision y penalización 5:1 se registran como controles. El 5:1 sigue siendo una hipótesis del grupo y no representa dinero.


In [ ]:
from importlib.metadata import version

print("Comprobando dependencias de la etapa 06...")
for package in ("pandas", "numpy", "scipy", "scikit-learn", "joblib", "lightgbm", "xgboost",
                "optuna", "mlflow", "databricks-sdk", "python-dotenv"):
    try:
        print(package, version(package))
    except Exception as error:
        raise RuntimeError(f"Falta la dependencia {package}; configura el entorno antes de continuar") from error

import optuna
from dataclasses import asdict

from proyecto.features.build import PACKAGE_E
from proyecto.models.candidates import make_lgbm
from proyecto.models.evaluate import score_folds
from proyecto.models.tuning import (
    TuningConfig, SearchSpace, run_search, make_search_folds,
    make_trial_pipeline, make_reference_pipeline, MlflowTrialLogger,
)

RUN_OPTIMIZATION = False  # Mantener False hasta autorizar la búsqueda real.
METRIC = "roc_auc"
FINALIST = "LightGBM con paquete E"
logistic_reference = make_reference_pipeline()  # Solo construcción, sin fit.
print("Candidato:", FINALIST, "| objetivo:", METRIC, "| dirección: maximize")
print("Paquete fijo:", list(PACKAGE_E))
print("Ejecución real habilitada:", RUN_OPTIMIZATION)


## 2. Conexión con el seguimiento

Se comprueba el destino configurado antes de Optuna. En Databricks se utiliza su servidor, sin respaldo silencioso a SQLite; localmente se conserva SQLite. La comprobación no crea experimentos. Al autorizar una búsqueda se crea un nombre único con fecha y UUID.

Cada ensayo terminado se registra una sola vez mediante MlflowTrialLogger: número, estado, parámetros, AUC disponible, duración y run_id. COMPLETE termina como FINISHED; FAIL como FAILED; las interrupciones capturadas como KILLED. La finalización de seguridad evita perder el ensayo interrumpido cuando Optuna omite el callback. Un error de seguimiento se propaga y detiene la búsqueda.

No se implementa recuperación del estudio. El cierre abrupto del proceso o una caída del servidor pueden impedir registrar el último ensayo. No hay garantía de exactamente una escritura entre procesos o reintentos de red; no se reintenta automáticamente la creación de runs.


In [ ]:
from datetime import datetime, timezone
from uuid import uuid4
import mlflow

from proyecto.tracking.mlflow_utils import (
    configure_mlflow, set_experiment, log_run, is_databricks, resolve_experiment_name,
)

ENABLE_MLFLOW = True
workspace_execution = is_databricks() or project_root.as_posix().startswith(("/Workspace/", "/Volumes/"))
TRACKING_MODE = os.environ.get("RISKVISION_TRACKING_MODE", "databricks" if workspace_execution else "local")
if workspace_execution and TRACKING_MODE != "databricks":
    raise ValueError("Dentro de Databricks usa TRACKING_MODE=databricks; no se admite respaldo local en esta etapa")
if workspace_execution and not ENABLE_MLFLOW:
    raise ValueError("La ejecución en Databricks requiere seguimiento habilitado")
tracking_location = experiment_name = experiment_id = None
if ENABLE_MLFLOW:
    tracking_location = configure_mlflow(backend=TRACKING_MODE)
    print("Seguimiento comprobado:", tracking_location)
    print("No se creó ningún experimento; los permisos de creación se comprobarán al autorizar la búsqueda.")


def prepare_tracking():
    """Crea un experimento nuevo al iniciar una búsqueda autorizada."""
    if not ENABLE_MLFLOW:
        return None, None, None
    if mlflow.active_run() is not None:
        raise RuntimeError("Hay un run activo: ciérralo antes de iniciar esta búsqueda")
    location = configure_mlflow(backend=TRACKING_MODE)
    name = "riskvision-06-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex
    requested_name = resolve_experiment_name(name)
    if mlflow.get_experiment_by_name(requested_name) is not None:
        raise RuntimeError("El nombre de experimento ya existe; no se reutiliza")
    full_name = set_experiment(requested_name)
    created = mlflow.get_experiment_by_name(full_name)
    if created is None:
        raise RuntimeError("No se pudo recuperar el experimento por su ruta completa")
    return location, full_name, created.experiment_id


En Databricks los runs se consultarán en el experimento del workspace cuya ruta indique la búsqueda. No se inicia un servidor MLflow local.

Solo en modo local, el seguimiento usa mlflow_local.db, ignorado por Git. Después de una ejecución autorizada puede consultarse con:

    mlflow ui --backend-store-uri sqlite:///mlflow_local.db

## 3. Reservar entidades que la búsqueda no verá

Se divide el entrenamiento original de forma **estratificada**, con semilla 7: 75 % para búsqueda (**18.000 clientes**) y 25 % de reserva interna (**6.000**). Son clientes distintos según ID, conservados para comprobar que no se solapan. No se sobrescriben los datos ni se guardan nuevas particiones.

La reserva no se usa para proponer hiperparámetros, detener ensayos o cambiar el presupuesto. Sin embargo, sus clientes ya participaron en las etapas anteriores. Es una comprobación interna de la búsqueda actual, no una prueba final independiente ni una validación cruzada anidada.


In [ ]:
RESERVE_SHARE = 0.25
RESERVE_SEED = 7
search_data, reserve_data = split_stratified(train, test_size=RESERVE_SHARE, seed=RESERVE_SEED)
assert_disjoint_ids(search_data, reserve_data)
assert len(search_data) == 3 * len(train) // 4
assert len(reserve_data) == len(train) // 4
assert set(search_data[ID_COL]) | set(reserve_data[ID_COL]) == set(train[ID_COL])
assert abs(search_data[TARGET].mean() - reserve_data[TARGET].mean()) < 0.005
X_search, y_search = make_xy(search_data)
X_reserve, y_reserve = make_xy(reserve_data)
display(pd.DataFrame({
    "clientes": [len(search_data), len(reserve_data)],
    "incumplimiento": [y_search.mean(), y_reserve.mean()],
}, index=["búsqueda", "reserva interna"]))


## 4. Espacio de búsqueda y presupuesto

Los rangos proceden de SearchSpace y son configurables antes de medir: 100–600 árboles, learning_rate 0,01–0,15, num_leaves 8–63, max_depth 3–10, min_child_samples 20–150, subsample y colsample_bytree 0,6–1, y reg_lambda 0,0001–10. Las hojas se limitan además a 2 elevado a la profundidad. La tasa de aprendizaje y la regularización positiva se exploran en escala logarítmica.

Se usan 5 pliegues estratificados por 3 repeticiones, semilla 42, exclusivamente dentro de los datos destinados a búsqueda. La configuración original se mide con esos mismos pliegues. No se compara directamente este AUC con el del torneo: allí se disponía de los 24.000 clientes.

El presupuesto inicial es **12 ensayos**, sin ejecución paralela de ensayos. Cambiar rangos o presupuesto después de consultar la reserva invalidaría su papel de comprobación.


In [ ]:
N_TRIALS = 12
SEARCH_SEED = 42
CONFIG = TuningConfig(n_trials=N_TRIALS, seed=SEARCH_SEED, n_splits=5, n_repeats=3, model_n_jobs=1)
SPACE = SearchSpace()
N_BOOTSTRAP = 1000
BOOTSTRAP_SEED = 42
print("Configuración:", asdict(CONFIG))
display(pd.DataFrame(asdict(SPACE), index=["mínimo", "máximo"]).T)
print("La reserva interna no participa en la búsqueda ni en la evaluación CV original.")


## 5. La búsqueda (un ensayo = una ejecución registrada)

La celda siguiente permanece inactiva mientras RUN_OPTIMIZATION sea False. Al habilitarla, primero mide la configuración original en los pliegues de búsqueda; después llama a run_search, que maximiza ROC AUC con pipelines nuevos y las mismas particiones reproducibles.

Se muestran todos los estados, el mejor ensayo completado, sus hiperparámetros y su AUC de validación. Los ensayos fallidos no reciben puntuaciones inventadas. Si no hay ensayos completos, se omite la comparación con la reserva.

El mejor AUC es un resultado de selección dentro de la búsqueda. No es una mejora demostrada ni un parámetro definitivo.


In [ ]:
study = original_cv = search_summary = comparison = bootstrap_result = None
trial_run_ids = {}
parameter_comparison = None
trial_logger = None
execution_started_at = None
execution_revision = None

# Mismos hiperparámetros de partida; solo se fijan semilla, hilos y determinismo como en tuning.py.
original_factory = lambda: make_lgbm(features=PACKAGE_E).set_params(
    model__random_state=CONFIG.seed, model__n_jobs=CONFIG.model_n_jobs,
    model__deterministic=True, model__force_col_wise=True,
)

if not RUN_OPTIMIZATION:
    print("Búsqueda pendiente de autorización: no se ajustaron modelos ni se creó un estudio.")
else:
    from proyecto.tracking.reporting import code_revision
    execution_started_at = datetime.now(timezone.utc).isoformat()
    execution_revision = code_revision(project_root,
        branch=os.environ.get("RISKVISION_CODE_BRANCH"), commit=os.environ.get("RISKVISION_CODE_COMMIT"))
    print("Procedencia del código:", execution_revision)
    if not execution_revision["branch"] or not execution_revision["commit"]:
        raise RuntimeError("Configura RISKVISION_CODE_BRANCH y RISKVISION_CODE_COMMIT si Git no está disponible")
    tracking_location, experiment_name, experiment_id = prepare_tracking()
    folds = make_search_folds(X_search, y_search, CONFIG)
    original_cv = score_folds(original_factory, X_search, y_search, folds)
    common_tags = {"stage": "06", "candidate": FINALIST, "training_sha256": TRAIN_SHA256,
                   "features": ",".join(PACKAGE_E), "status": "exploratory",
                   "code_branch": execution_revision["branch"] or "unknown",
                   "code_commit": execution_revision["commit"] or "unknown",
                   "execution_started_at": execution_started_at}
    if experiment_id is not None:
        log_run("original-cv", {**original_factory().named_steps["model"].get_params(),
                               **asdict(CONFIG), "reserve_seed": RESERVE_SEED},
                {"roc_auc_cv": float(original_cv[METRIC].mean()),
                 "avg_precision_mean": float(original_cv["avg_precision"].mean()),
                 "penalty_mean": float(original_cv["penalty"].mean())},
                tags={**common_tags, "role": "baseline"})
        trial_logger = MlflowTrialLogger(experiment_id,
            params={**original_factory().named_steps["model"].get_params(), **asdict(CONFIG)}, tags=common_tags)
        trial_run_ids = trial_logger.run_ids  # También accesible si se interrumpe la celda.
    study = run_search(X_search, y_search, config=CONFIG, space=SPACE, trial_logger=trial_logger)
    assert study.direction == optuna.study.StudyDirection.MAXIMIZE
    study.set_user_attr("reserve_seed", RESERVE_SEED)
    study.set_user_attr("reserve_share", RESERVE_SHARE)
    study.set_user_attr("training_sha256", TRAIN_SHA256)
    display(study.trials_dataframe())
    completed = [trial for trial in study.trials if trial.state == optuna.trial.TrialState.COMPLETE]
    if completed:
        best = study.best_trial
        print("Mejor ensayo completado:", best.number)
        print("Mejores hiperparámetros candidatos:", best.params)
        original_parameters = original_factory().named_steps["model"].get_params()
        parameter_comparison = pd.DataFrame({
            "original": {name: original_parameters[name] for name in best.params},
            "candidato Optuna": best.params,
        })
        display(parameter_comparison)
        search_summary = pd.DataFrame({
            "ROC AUC CV": [original_cv[METRIC].mean(), best.value],
        }, index=["LightGBM original", "LightGBM seleccionado por Optuna"])
        display(search_summary)
        print("El valor seleccionado tiene optimismo; no demuestra superioridad.")
    else:
        print("Todos los ensayos fallaron: no hay mejores parámetros ni comparación de reserva.")
    if experiment_id is not None:
        assert len(trial_run_ids) == len(study.trials)
        print("Experimento nuevo:", experiment_name, "| ensayos registrados:", len(trial_run_ids))


## 6. La comprobación honesta

Se ajustan dos pipelines independientes —hiperparámetros de partida y seleccionados— **solo con los 18.000 clientes de búsqueda**. Ambos predicen los mismos 6.000 clientes de reserva. No se ajusta ningún componente con la reserva.

ROC AUC no es un costo por fila. El bootstrap pareado remuestrea los mismos clientes para ambas predicciones y recalcula ambos AUC en cada muestra. La diferencia se define como **AUC optimizado − AUC original**, por lo que un valor positivo favorece al optimizado. Una fila representa un cliente único; la función rechaza ID repetidos. Las muestras con una sola clase se omiten y se informa cuántas fueron válidas.

**Criterio previo de lectura:** un intervalo completamente por encima de cero sugeriría una mejora condicional en esta reserva; si contiene cero, no separa la mejora del ruido de esta comparación. No se promueve automáticamente ningún modelo. Incluso si excluye cero, el intervalo no elimina el sesgo de selección anterior, no incorpora la incertidumbre de volver a entrenar o afinar y no constituye evidencia de generalización independiente.


In [ ]:
from sklearn.metrics import roc_auc_score


def paired_auc_bootstrap(y_true, original_proba, optimized_proba, client_ids, n_bootstrap=1000, seed=42):
    """Intervalo percentil pareado, condicionado a los modelos ya ajustados."""
    labels = np.asarray(y_true)
    original = np.asarray(original_proba, dtype=float)
    optimized = np.asarray(optimized_proba, dtype=float)
    ids = np.asarray(client_ids)
    if any(values.ndim != 1 for values in (labels, original, optimized, ids)):
        raise ValueError("Se requieren vectores unidimensionales")
    if not (len(labels) == len(original) == len(optimized) == len(ids)) or len(labels) < 2:
        raise ValueError("Longitudes incompatibles o muestra insuficiente")
    assert_unique_ids(pd.DataFrame({ID_COL: ids}))
    if not pd.Series(labels).isin([0, 1]).all() or set(labels) != {0, 1}:
        raise ValueError("Se requieren ambas clases 0 y 1 sin vacíos")
    if not all(np.isfinite(values).all() and ((values >= 0) & (values <= 1)).all()
               for values in (original, optimized)):
        raise ValueError("Probabilidades no finitas o fuera de [0, 1]")
    if not isinstance(n_bootstrap, (int, np.integer)) or isinstance(n_bootstrap, (bool, np.bool_)) or n_bootstrap < 2:
        raise ValueError("n_bootstrap debe ser un entero >= 2")
    rng = np.random.default_rng(seed)
    differences = []
    for _ in range(n_bootstrap):
        positions = rng.integers(0, len(labels), size=len(labels))
        sampled = labels[positions]
        if len(np.unique(sampled)) < 2:
            continue
        differences.append(roc_auc_score(sampled, optimized[positions])
                           - roc_auc_score(sampled, original[positions]))
    if len(differences) < 2:
        raise ValueError("No hay suficientes remuestras con ambas clases")
    low, high = np.percentile(differences, [2.5, 97.5])
    return {"delta_auc": float(roc_auc_score(labels, optimized) - roc_auc_score(labels, original)),
            "low": float(low), "high": float(high), "valid_draws": len(differences),
            "attempted_draws": int(n_bootstrap)}


if not RUN_OPTIMIZATION or study is None or not any(trial.state == optuna.trial.TrialState.COMPLETE for trial in study.trials):
    print("Comparación de reserva pendiente: todavía no hay búsqueda válida ejecutada.")
else:
    original_model = original_factory().fit(X_search, y_search)
    optimized_model = make_trial_pipeline(study.best_params, seed=CONFIG.seed,
                                           model_n_jobs=CONFIG.model_n_jobs).fit(X_search, y_search)
    original_proba = original_model.predict_proba(X_reserve)[:, 1]
    optimized_proba = optimized_model.predict_proba(X_reserve)[:, 1]
    original_auc = roc_auc_score(y_reserve, original_proba)
    optimized_auc = roc_auc_score(y_reserve, optimized_proba)
    comparison = pd.DataFrame({"ROC AUC reserva interna": [original_auc, optimized_auc]},
                              index=["LightGBM original", "LightGBM seleccionado por Optuna"])
    bootstrap_result = paired_auc_bootstrap(y_reserve, original_proba, optimized_proba,
                                           reserve_data[ID_COL], n_bootstrap=N_BOOTSTRAP, seed=BOOTSTRAP_SEED)
    display(comparison)
    print(f"Diferencia de AUC (optimizado - original): {bootstrap_result['delta_auc']:+.6f}")
    print(f"Intervalo percentil 95 %: [{bootstrap_result['low']:+.6f}, {bootstrap_result['high']:+.6f}]")
    print("Remuestras válidas:", bootstrap_result["valid_draws"], "/", bootstrap_result["attempted_draws"])
    if bootstrap_result["low"] > 0:
        print("Sugiere mejora condicional en esta reserva; requiere revisión, no promoción automática.")
    else:
        print("El intervalo no respalda una mejora positiva clara en esta reserva.")
    if experiment_id is not None:
        log_run("internal-reserve-comparison", {"bootstrap_seed": BOOTSTRAP_SEED, "n_bootstrap": N_BOOTSTRAP},
                {"original_auc": float(original_auc), "optimized_auc": float(optimized_auc),
                 **{key: float(value) for key, value in bootstrap_result.items()}},
                tags={"stage": "06", "role": "internal_comparison", "status": "pending_review"})


**Resultado y decisión:** pendientes de ejecutar y revisar. La celda anterior calculará la diferencia y el intervalo, sin inventar resultados. La regla se definió antes de ejecutar; positivo favorece al optimizado porque se compara AUC, no costo.

La reserva interna se consulta una vez para esta comparación. No debe reutilizarse para retocar rangos, variables, calibración o presupuesto. Si se toman nuevas decisiones con ella, habrá que reconocer esa adaptación y replantear la evaluación.

## 7. Preparar la decisión para revisión

Se prepara un informe provisional con fecha UTC, rama, commit, estado del código, versiones, configuración, ensayos y sus run_id, métricas y limitaciones. Si Git no está disponible en Databricks, indicar RISKVISION_CODE_BRANCH y RISKVISION_CODE_COMMIT antes de buscar. Se registra como artefacto en un run nuevo de MLflow; repetir el informe crea otro run y no sobrescribe los anteriores. Sin seguimiento queda solo en memoria. No se exportan modelos, no se crea un archivo de parámetros definitivos y no se cambia FINAL_FEATURES ni la configuración compartida del torneo. La aprobación del grupo y una evaluación posterior siguen pendientes.


In [ ]:
from proyecto.tracking.reporting import build_provisional_report, log_provisional_report

review_summary = None
provisional_report_run_id = None
if not RUN_OPTIMIZATION or study is None:
    print("Informe pendiente: no se ejecutó una búsqueda autorizada.")
else:
    review_summary = build_provisional_report(
        revision=execution_revision, config=asdict(CONFIG), space=asdict(SPACE), study=study,
        original_cv_auc=float(original_cv[METRIC].mean()),
        comparison=None if comparison is None else comparison.to_dict(), bootstrap=bootstrap_result,
        trial_run_ids=trial_run_ids, training_sha256=TRAIN_SHA256,
        reserve={"seed": RESERVE_SEED, "share": RESERVE_SHARE,
                 "search_clients": len(search_data), "reserve_clients": len(reserve_data)},
        versions={package: version(package) for package in (
            "pandas", "numpy", "scipy", "scikit-learn", "lightgbm", "xgboost", "optuna", "mlflow", "joblib")},
        experiment_name=experiment_name, execution_started_at=execution_started_at,
    )
    display(review_summary)
    if experiment_id is not None:
        provisional_report_run_id = log_provisional_report(review_summary, experiment_id)
        print("Informe provisional en run nuevo:", provisional_report_run_id,
              "| artefacto: reports/provisional.json")
    else:
        print("MLflow deshabilitado: informe provisional solo en memoria.")
    print("Pendiente de revisión: no se promovieron parámetros ni se exportaron modelos.")


## 8. Comprobar el seguimiento

Después de autorizar y ejecutar, revisar el experimento nuevo que indique la sección 5 en MLflow del workspace o local, según el destino verificado. Cada ensayo debe conservar sus parámetros, AUC y estado Optuna; los runs de referencia CV y comparación interna se distinguen por la etiqueta role. No se modifican experimentos anteriores.

El registro de ensayos es progresivo; los callbacks repetidos no crean runs duplicados. Los informes provisionales se guardan en runs nuevos identificados con role=provisional_report y status=pending_review. La recuperación de una búsqueda interrumpida y la persistencia del estudio Optuna quedan pendientes. Los UUID identifican ejecuciones; las semillas, las particiones, la huella del entrenamiento y las versiones del entorno permiten auditar la reproducción numérica.

## Entregable de esta etapa

- [x] Integración preparada con tuning.py y paquete E fijo; dirección maximize.
- [x] Presupuesto inicial de 12 ensayos y semillas configurables.
- [x] Reserva interna estratificada del 25 % y comprobaciones de ID.
- [x] Comparación original frente a optimizado y bootstrap pareado de AUC implementados.
- [x] Registro MLflow preparado con experimento nuevo y sin exportación definitiva.
- [ ] Autorización para activar RUN_OPTIMIZATION y ejecutar sobre datos reales.
- [ ] Revisar resultados, fallos, intervalo y limitaciones con el grupo.
- [ ] Aprobar una decisión antes de completar informes o exportar parámetros/modelos.

Ninguna casilla de implementación equivale a una búsqueda real ya ejecutada ni demuestra mejora.
